<img src="image.png" width="25%">

In [ ]:
"""
Health-Aware High-Frequency Trading for Battery Energy Storage Systems
Implements both MILP (Gurobi) and DP approaches from the paper

Usage:
    python script.py path/to/orderbook.csv

CSV Format Required:
    - side: BUY or SELL
    - start: Delivery period start (ISO 8601)
    - transaction: Order submission time (ISO 8601)
    - validity: Order expiry time (ISO 8601, optional)
    - price: Order price (€/MWh)
    - quantity: Order quantity (MWh)
"""

import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass
import time


@dataclass
class BatteryParams:
    """Battery system parameters"""
    power_max: float = 10.0  # MW (charging limit)
    power_min: float = -10.0  # MW (discharging limit)
    energy_capacity: float = 10.0  # MWh
    energy_min: float = 0.0  # MWh
    energy_max: float = 10.0  # MWh
    eta_charge: float = 0.95  # Charging efficiency
    eta_discharge: float = 0.95  # Discharging efficiency
    replacement_cost: float = 300000.0  # €/MWh
    num_segments: int = 16  # Piecewise linear segments for degradation


@dataclass
class MarketParams:
    """Market parameters"""
    trading_fee: float = 0.09  # €/MWh
    min_trading_unit: float = 0.1  # MWh
    time_interval: float = 1.0  # hours


@dataclass
class Order:
    """Limit order book entry"""
    price: float  # €/MWh
    quantity: float  # MWh
    is_buy: bool  # True for buy orders, False for sell orders


class DegradationModel:
    """Battery degradation model with piecewise linear approximation"""
    
    def __init__(self, battery: BatteryParams):
        self.battery = battery
        self.J = battery.num_segments
        self.segment_costs = self._compute_segment_costs()
    
    def stress_function(self, delta: float) -> float:
        """Cycle depth stress function Φ(δ) for Li(NiMnCo)O2 cells"""
        return 5.24e-4 * (delta ** 2.03)
    
    def _compute_segment_costs(self) -> np.ndarray:
        """Compute marginal cost for each cycle depth segment"""
        costs = np.zeros(self.J)
        R = self.battery.replacement_cost
        eta_dis = self.battery.eta_discharge
        E_rate = self.battery.energy_capacity
        
        for j in range(self.J):
            phi_upper = self.stress_function((j + 1) / self.J)
            phi_lower = self.stress_function(j / self.J)
            costs[j] = (R * self.J / (eta_dis * E_rate)) * (phi_upper - phi_lower)
        
        return costs
    
    def compute_degradation_cost(self, soc: float, discharge_energy: float) -> float:
        """Compute degradation cost for a discharge action"""
        if discharge_energy <= 0:
            return 0.0
        
        segment_width = self.battery.energy_capacity / self.J
        n_full = int(soc / segment_width)
        frac_fill = (soc / segment_width) - n_full
        
        cost = 0.0
        remaining = discharge_energy
        
        for j in range(self.J):
            if remaining <= 0:
                break
            
            # Determine energy in segment j
            if j < n_full:
                e_j = segment_width
            elif j == n_full:
                e_j = frac_fill * segment_width
            else:
                e_j = 0.0
            
            # Discharge from this segment
            discharged = min(e_j, remaining)
            cost += self.segment_costs[j] * discharged
            remaining -= discharged
        
        return cost


class HealthAwareMILP:
    """MILP formulation for health-aware battery trading"""
    
    def __init__(self, battery: BatteryParams, market: MarketParams, 
                 degradation: DegradationModel):
        self.battery = battery
        self.market = market
        self.degradation = degradation
    
    def solve(self, orders: Dict[int, List[Order]], initial_soc: float,
              time_horizon: List[int], committed_power: Optional[Dict[int, float]] = None,
              final_soc: Optional[float] = None,
              initial_segment_energy: Optional[Dict[int, float]] = None) -> Dict:
        """
        Solve the health-aware intrinsic optimization problem using MILP
        Implements equations (11a)-(11w) exactly as in the paper
        
        Args:
            orders: Dictionary mapping time period to list of orders
            initial_soc: Initial state of charge E^0 (MWh)
            time_horizon: List of time periods to optimize
            committed_power: Previously committed power f_t^0 for each period
            final_soc: Required final state of charge E^final
            initial_segment_energy: Initial energy e_j^0 in each segment
        
        Returns:
            Dictionary with solution details
        """
        T = len(time_horizon)
        J = self.degradation.J
        M = self.market.time_interval
        u = self.market.min_trading_unit
        nu = self.market.trading_fee  # ν in the paper
        
        if committed_power is None:
            committed_power = {t: 0.0 for t in time_horizon}
        
        # Initial segment energy distribution (uniform if not specified)
        if initial_segment_energy is None:
            initial_segment_energy = {j: initial_soc / J for j in range(J)}
        
        model = gp.Model("HealthAwareBESS")
        model.setParam('OutputFlag', 0)  # Suppress output
        
        # ==================== DECISION VARIABLES ====================
        
        # k_i: Integer multiplier (blocks) for order i (11w)
        k = {}
        q = {}  # q_i = k_i * u (11c)
        for t in time_horizon:
            for i, order in enumerate(orders.get(t, [])):
                max_blocks = int(order.quantity / u)
                k[i, t] = model.addVar(vtype=GRB.INTEGER, lb=0, ub=max_blocks,
                                       name=f"k_{i}_{t}")
                q[i, t] = model.addVar(lb=0, ub=order.quantity, name=f"q_{i}_{t}")
        
        # α_t: Binary variable for operating mode (1=charge, 0=discharge) (11w)
        alpha = {t: model.addVar(vtype=GRB.BINARY, name=f"alpha_{t}") 
                 for t in time_horizon}
        
        # p_t^{ch,j}, p_t^{dis,j}: Charging and discharging power per segment
        p_ch = {(t, j): model.addVar(lb=0, name=f"p_ch_{t}_{j}")
                for t in time_horizon for j in range(J)}
        p_dis = {(t, j): model.addVar(lb=0, name=f"p_dis_{t}_{j}")
                 for t in time_horizon for j in range(J)}
        
        # e_{t,j}: Energy stored in segment j at time t (11r)
        e_bar_j = self.battery.energy_capacity / J  # ē_j
        e = {(t, j): model.addVar(lb=0, ub=e_bar_j, name=f"e_{t}_{j}")
             for t in time_horizon for j in range(J)}
        
        # s_t: Total state of charge at time t (11l)
        s = {t: model.addVar(lb=self.battery.energy_min, ub=self.battery.energy_max,
                            name=f"s_{t}") for t in time_horizon}
        
        # i_t, w_t: Total charging and discharging power
        i_var = {t: model.addVar(lb=0, name=f"i_{t}") for t in time_horizon}
        w_var = {t: model.addVar(lb=0, name=f"w_{t}") for t in time_horizon}
        
        # f_t^+, f_t^-: Total buying and selling power
        f_plus = {t: model.addVar(lb=0, name=f"f_plus_{t}") for t in time_horizon}
        f_minus = {t: model.addVar(lb=0, name=f"f_minus_{t}") for t in time_horizon}
        
        # f_t: Net power exchanged with grid (11g)
        f = {t: model.addVar(lb=M*self.battery.power_min, 
                            ub=M*self.battery.power_max,
                            name=f"f_{t}") for t in time_horizon}
        
        model.update()
        
        # ==================== OBJECTIVE FUNCTION (11a) ====================
        obj = 0
        for t in time_horizon:
            # Revenue from selling (orders in O_t^-)
            for i, order in enumerate(orders.get(t, [])):
                if not order.is_buy:  # Sell orders in O_t^-
                    obj += (order.price - nu) * q[i, t]
            
            # Cost from buying (orders in O_t^+)
            for i, order in enumerate(orders.get(t, [])):
                if order.is_buy:  # Buy orders in O_t^+
                    obj -= (order.price + nu) * q[i, t]
            
            # Degradation cost
            for j in range(J):
                obj -= M * self.degradation.segment_costs[j] * p_dis[t, j]
        
        model.setObjective(obj, GRB.MAXIMIZE)
        
        # ==================== CONSTRAINTS ====================
        
        for t_idx, t in enumerate(time_horizon):
            
            # (11b): 0 ≤ q_i ≤ Q_i, ∀i ∈ O_t, ∀t ∈ T
            for i, order in enumerate(orders.get(t, [])):
                model.addConstr(q[i, t] >= 0)
                model.addConstr(q[i, t] <= order.quantity)
                
                # (11c): q_i = k_i * u
                model.addConstr(q[i, t] == k[i, t] * u)
            
            # (11d): f_t^+ = Σ_{i∈O_t^+} q_i
            buy_orders = [i for i, order in enumerate(orders.get(t, [])) if order.is_buy]
            if buy_orders:
                model.addConstr(f_plus[t] == gp.quicksum(q[i, t] for i in buy_orders))
            else:
                model.addConstr(f_plus[t] == 0)
            
            # (11e): f_t^- = Σ_{i∈O_t^-} q_i
            sell_orders = [i for i, order in enumerate(orders.get(t, [])) if not order.is_buy]
            if sell_orders:
                model.addConstr(f_minus[t] == gp.quicksum(q[i, t] for i in sell_orders))
            else:
                model.addConstr(f_minus[t] == 0)
            
            # (11f): f_t = f_t^0 + f_t^+ - f_t^-
            model.addConstr(f[t] == committed_power[t] + f_plus[t] - f_minus[t])
            
            # (11g): f_t ∈ [M*f, M*f̄]
            model.addConstr(f[t] >= M * self.battery.power_min)
            model.addConstr(f[t] <= M * self.battery.power_max)
            
            # (11h): f_t = M(i_t - w_t)
            model.addConstr(f[t] == M * (i_var[t] - w_var[t]))
            
            # (11i): i_t ∈ [0, α_t * f̄]
            model.addConstr(i_var[t] >= 0)
            model.addConstr(i_var[t] <= alpha[t] * self.battery.power_max)
            
            # (11j): w_t ∈ [0, -(1-α_t)*f]
            model.addConstr(w_var[t] >= 0)
            model.addConstr(w_var[t] <= (1 - alpha[t]) * (-self.battery.power_min))
            
            # (11m): w_t = Σ_{j=1}^J p_t^{dis,j}
            model.addConstr(w_var[t] == gp.quicksum(p_dis[t, j] for j in range(J)))
            
            # (11n): i_t = Σ_{j=1}^J p_t^{ch,j}
            model.addConstr(i_var[t] == gp.quicksum(p_ch[t, j] for j in range(J)))
            
            # (11o): e_{t,j} - e_{t-1,j} = M(p_t^{ch,j}η^+ - p_t^{dis,j}/η^-)
            for j in range(J):
                if t_idx == 0:
                    # (11s): e_{0,j} = e_j^0
                    e_prev = initial_segment_energy[j]
                else:
                    e_prev = e[time_horizon[t_idx-1], j]
                
                model.addConstr(
                    e[t, j] == e_prev + M * (
                        p_ch[t, j] * self.battery.eta_charge - 
                        p_dis[t, j] / self.battery.eta_discharge
                    )
                )
            
            # (11p): p_t^{ch,j} ≥ 0, p_t^{dis,j} ≥ 0
            for j in range(J):
                model.addConstr(p_ch[t, j] >= 0)
                model.addConstr(p_dis[t, j] >= 0)
            
            # (11q): s_t = Σ_{j=1}^J e_{t,j}
            model.addConstr(s[t] == gp.quicksum(e[t, j] for j in range(J)))
            
            # (11r): 0 ≤ e_{t,j} ≤ ē_j (already in variable bounds)
            
            # (11k): s_t = s_{t-1} + M(η^+i_t - 1/η^- w_t)
            if t_idx == 0:
                # s_0 = E^0 = Σ_{j=1}^J e_{0,j}
                model.addConstr(s[t] == initial_soc)
            else:
                s_prev = s[time_horizon[t_idx-1]]
                model.addConstr(
                    s[t] == s_prev + M * (
                        self.battery.eta_charge * i_var[t] - 
                        w_var[t] / self.battery.eta_discharge
                    )
                )
            
            # (11l): s_t ∈ [E^min, E^max] (already in variable bounds)
        
        # (11t): Σ_{j=1}^J e_{T,j} ≥ E^final
        if final_soc is not None:
            model.addConstr(
                gp.quicksum(e[time_horizon[-1], j] for j in range(J)) >= final_soc
            )
        
        # Solve
        start_time = time.time()
        model.optimize()
        solve_time = time.time() - start_time
        
        if model.status != GRB.OPTIMAL:
            return {
                'status': 'infeasible',
                'solve_time': solve_time
            }
        
        # Extract solution
        solution = {
            'status': 'optimal',
            'objective': model.objVal,
            'solve_time': solve_time,
            'trades': {},
            'soc': {},
            'power': {}
        }
        
        for t in time_horizon:
            solution['soc'][t] = s[t].X
            solution['power'][t] = f[t].X / M
            solution['trades'][t] = []
            
            for i, order in enumerate(orders.get(t, [])):
                if k[i, t].X > 0.5:  # Account for numerical tolerance
                    solution['trades'][t].append({
                        'order_idx': i,
                        'blocks': int(k[i, t].X),
                        'quantity': k[i, t].X * u,
                        'price': order.price,
                        'is_buy': order.is_buy
                    })
        
        return solution


class HealthAwareDP:
    """Dynamic Programming formulation for health-aware battery trading"""
    
    def __init__(self, battery: BatteryParams, market: MarketParams,
                 degradation: DegradationModel, num_states: int = 51):
        self.battery = battery
        self.market = market
        self.degradation = degradation
        self.num_states = num_states
        
        # Create state grid
        self.state_grid = np.linspace(battery.energy_min, battery.energy_max, 
                                     num_states)
        self.delta_s = self.state_grid[1] - self.state_grid[0]
    
    def _get_feasible_actions(self, state: float, orders: List[Order]) -> List[Tuple[float, float]]:
        """
        Get feasible actions (energy change) from current state
        Returns list of (action, profit) tuples
        """
        actions = []
        u = self.market.min_trading_unit
        M = self.market.time_interval
        
        # Buy orders (charging)
        for order in orders:
            if order.is_buy:
                max_energy = min(
                    order.quantity,
                    M * self.battery.power_max,
                    self.battery.energy_max - state
                )
                
                # Discretize to trading units
                max_blocks = int(max_energy / u)
                for blocks in range(1, max_blocks + 1):
                    energy = blocks * u
                    energy_stored = energy * self.battery.eta_charge
                    profit = -(order.price + self.market.trading_fee) * energy
                    
                    if state + energy_stored <= self.battery.energy_max:
                        actions.append((energy_stored, profit))
        
        # Sell orders (discharging)
        for order in orders:
            if not order.is_buy:
                max_energy = min(
                    order.quantity,
                    M * (-self.battery.power_min),
                    state - self.battery.energy_min
                )
                
                max_blocks = int(max_energy / u)
                for blocks in range(1, max_blocks + 1):
                    energy = blocks * u
                    energy_discharged = energy / self.battery.eta_discharge
                    profit = (order.price - self.market.trading_fee) * energy
                    
                    if state - energy_discharged >= self.battery.energy_min:
                        actions.append((-energy_discharged, profit))
        
        # Add idle action
        actions.append((0.0, 0.0))
        
        return actions
    
    def _interpolate_value(self, value_func: np.ndarray, state: float) -> float:
        """Linear interpolation of value function"""
        if state <= self.state_grid[0]:
            return value_func[0]
        if state >= self.state_grid[-1]:
            return value_func[-1]
        
        # Find bracketing indices
        idx = np.searchsorted(self.state_grid, state)
        s_low, s_high = self.state_grid[idx-1], self.state_grid[idx]
        v_low, v_high = value_func[idx-1], value_func[idx]
        
        # Linear interpolation
        weight = (state - s_low) / (s_high - s_low)
        return v_low * (1 - weight) + v_high * weight
    
    def solve(self, orders: Dict[int, List[Order]], initial_soc: float,
              time_horizon: List[int], final_soc: Optional[float] = None,
              phi: float = 0.0) -> Dict:
        """
        Solve using Dynamic Programming
        
        Args:
            orders: Dictionary mapping time period to list of orders
            initial_soc: Initial state of charge
            time_horizon: List of time periods
            final_soc: Required final SoC
            phi: Bid-ask spread penalty parameter
        
        Returns:
            Dictionary with solution details
        """
        T = len(time_horizon)
        
        # Initialize value functions
        V = np.zeros((T + 1, self.num_states))
        policy = [[None for _ in range(self.num_states)] for _ in range(T)]
        
        # Backward pass
        start_time = time.time()
        
        for t_idx in range(T - 1, -1, -1):
            t = time_horizon[t_idx]
            
            for s_idx, s in enumerate(self.state_grid):
                # Check final SoC constraint at last stage
                if t_idx == T - 1 and final_soc is not None:
                    if s < final_soc - 1e-6:
                        V[t_idx, s_idx] = -1e10
                        continue
                
                best_value = -np.inf
                best_action = None
                
                # Evaluate all feasible actions
                actions = self._get_feasible_actions(s, orders.get(t, []))
                
                for action, profit in actions:
                    next_state = s + action
                    
                    # Check state bounds
                    if next_state < self.battery.energy_min - 1e-6 or \
                       next_state > self.battery.energy_max + 1e-6:
                        continue
                    
                    # Compute degradation cost (only for discharge)
                    deg_cost = 0.0
                    if action < 0:
                        deg_cost = self.degradation.compute_degradation_cost(
                            s, -action)
                    
                    # Compute bid-ask spread penalty if phi > 0
                    spread_penalty = 0.0
                    if phi > 0 and len(orders.get(t, [])) > 0:
                        buy_prices = [o.price for o in orders[t] if o.is_buy]
                        sell_prices = [o.price for o in orders[t] if not o.is_buy]
                        if buy_prices and sell_prices:
                            bid_ask_spread = (min(buy_prices) - max(sell_prices)) / \
                                           ((min(buy_prices) + max(sell_prices)) / 2)
                            spread_penalty = phi * bid_ask_spread * abs(action)
                    
                    # Get next value
                    next_value = self._interpolate_value(V[t_idx + 1], next_state)
                    
                    # Total value
                    total_value = profit - deg_cost - spread_penalty + next_value
                    
                    if total_value > best_value:
                        best_value = total_value
                        best_action = (action, profit, deg_cost)
                
                V[t_idx, s_idx] = best_value
                policy[t_idx][s_idx] = best_action
        
        solve_time = time.time() - start_time
        
        # Forward pass to extract policy
        solution = {
            'status': 'optimal',
            'solve_time': solve_time,
            'soc': {},
            'power': {},
            'trades': {},
            'total_profit': 0.0,
            'total_degradation': 0.0
        }
        
        # Find closest state to initial SoC
        s_idx = np.argmin(np.abs(self.state_grid - initial_soc))
        current_soc = initial_soc
        
        solution['objective'] = self._interpolate_value(V[0], initial_soc)
        
        for t_idx, t in enumerate(time_horizon):
            s_idx = np.argmin(np.abs(self.state_grid - current_soc))
            
            if policy[t_idx][s_idx] is not None:
                action, profit, deg_cost = policy[t_idx][s_idx]
                current_soc += action
                
                solution['soc'][t] = current_soc
                solution['power'][t] = action / self.market.time_interval
                solution['total_profit'] += profit
                solution['total_degradation'] += deg_cost
            else:
                solution['soc'][t] = current_soc
                solution['power'][t] = 0.0
        
        return solution


class OrderBookLoader:
    """Load and process EPEX SPOT-style order book data from CSV"""
    
    def __init__(self, csv_path: str):
        """
        Initialize order book loader
        
        Args:
            csv_path: Path to CSV file with order book data
        """
        self.csv_path = csv_path
        self.data = None
        self.load_data()
    
    def load_data(self):
        """Load order book data from CSV"""
        import pandas as pd
        
        print(f"Loading order book data from {self.csv_path}...")
        self.data = pd.read_csv(self.csv_path)
        
        # Convert timestamps
        self.data['transaction'] = pd.to_datetime(self.data['transaction'])
        self.data['start'] = pd.to_datetime(self.data['start'])
        
        # Handle validity (can be NaN)
        if 'validity' in self.data.columns:
            self.data['validity'] = pd.to_datetime(self.data['validity'], errors='coerce')
        
        print(f"Loaded {len(self.data)} orders")
        print(f"Date range: {self.data['start'].min()} to {self.data['start'].max()}")
        print(f"Products: {self.data['start'].nunique()} unique delivery periods")
    
    def get_orders_at_time(self, current_time: pd.Timestamp, 
                          delivery_start: pd.Timestamp) -> List[Order]:
        """
        Get active orders at a specific time for a delivery period
        
        Args:
            current_time: Current market time (when we're making decision)
            delivery_start: Delivery period start time
        
        Returns:
            List of Order objects
        """
        # Filter orders for this delivery period
        mask = self.data['start'] == delivery_start
        
        # Order must have been submitted before current time
        mask &= self.data['transaction'] <= current_time
        
        # Order must still be valid (if validity exists)
        if 'validity' in self.data.columns:
            # NaN validity means order stays active
            mask &= (self.data['validity'].isna() | (self.data['validity'] > current_time))
        
        active_orders = self.data[mask]
        
        # Convert to Order objects
        orders = []
        for _, row in active_orders.iterrows():
            is_buy = row['side'] == 'BUY'
            orders.append(Order(
                price=row['price'],
                quantity=row['quantity'],
                is_buy=is_buy
            ))
        
        return orders
    
    def get_order_book_snapshot(self, current_time: pd.Timestamp) -> Dict[pd.Timestamp, List[Order]]:
        """
        Get order book snapshot for all active products at current time
        
        Args:
            current_time: Current market time
        
        Returns:
            Dictionary mapping delivery start time to list of orders
        """
        # Get all delivery periods that haven't happened yet
        future_products = self.data[self.data['start'] > current_time]['start'].unique()
        
        order_book = {}
        for product in sorted(future_products):
            orders = self.get_orders_at_time(current_time, product)
            if orders:  # Only include if there are active orders
                order_book[product] = orders
        
        return order_book
    
    def get_trading_timeline(self) -> List[pd.Timestamp]:
        """Get all unique transaction times for backtesting"""
        return sorted(self.data['transaction'].unique())
    
    def get_best_bid_ask(self, orders: List[Order]) -> Tuple[float, float, float]:
        """
        Get best bid, best ask, and spread from order list
        
        Returns:
            (best_bid, best_ask, spread)
        """
        buy_orders = [o for o in orders if o.is_buy]
        sell_orders = [o for o in orders if not o.is_buy]
        
        best_bid = max([o.price for o in buy_orders]) if buy_orders else 0.0
        best_ask = min([o.price for o in sell_orders]) if sell_orders else 1e6
        
        if best_bid > 0 and best_ask < 1e6:
            spread = (best_ask - best_bid) / ((best_ask + best_bid) / 2)
        else:
            spread = 0.0
        
        return best_bid, best_ask, spread


class RollingIntrinsicBacktest:
    """Backtest the Rolling Intrinsic policy on historical order book data"""
    
    def __init__(self, battery: BatteryParams, market: MarketParams,
                 degradation: DegradationModel, use_dp: bool = True,
                 num_states: int = 51, phi: float = 0.0):
        """
        Initialize backtesting framework
        
        Args:
            battery: Battery parameters
            market: Market parameters
            degradation: Degradation model
            use_dp: Use DP (True) or MILP (False)
            num_states: Number of states for DP
            phi: Bid-ask spread penalty parameter
        """
        self.battery = battery
        self.market = market
        self.degradation = degradation
        self.use_dp = use_dp
        self.phi = phi
        
        if use_dp:
            self.solver = HealthAwareDP(battery, market, degradation, num_states)
        else:
            self.solver = HealthAwareMILP(battery, market, degradation)
        
        # Tracking
        self.soc_history = []
        self.profit_history = []
        self.degradation_history = []
        self.solve_time_history = []
        self.trade_history = []
    
    def run(self, order_book_loader: OrderBookLoader, 
            initial_soc: float = 5.0,
            max_horizon_hours: int = 24,
            update_frequency: str = 'hourly') -> Dict:
        """
        Run backtest on order book data
        
        Args:
            order_book_loader: Loaded order book data
            initial_soc: Initial battery state of charge (MWh)
            max_horizon_hours: Maximum optimization horizon in hours
            update_frequency: How often to re-optimize ('every_update', 'hourly', 'minutely')
        
        Returns:
            Dictionary with backtest results
        """
        import pandas as pd
        
        print(f"\n{'='*70}")
        print(f"Running Rolling Intrinsic Backtest")
        print(f"Solver: {'DP' if self.use_dp else 'MILP'}")
        print(f"Update frequency: {update_frequency}")
        print(f"Phi penalty: {self.phi}")
        print(f"{'='*70}\n")
        
        current_soc = initial_soc
        cumulative_profit = 0.0
        cumulative_degradation = 0.0
        
        # Get trading timeline
        timeline = order_book_loader.get_trading_timeline()
        
        # Filter based on update frequency
        if update_frequency == 'hourly':
            timeline = [t for t in timeline if t.minute == 0]
        elif update_frequency == 'minutely':
            timeline = [t for t in timeline if t.second == 0]
        
        print(f"Timeline: {len(timeline)} decision points")
        print(f"Start: {timeline[0]}")
        print(f"End: {timeline[-1]}\n")
        
        total_solves = 0
        
        for idx, current_time in enumerate(timeline):
            if idx % 100 == 0:
                print(f"Progress: {idx}/{len(timeline)} ({100*idx/len(timeline):.1f}%) | "
                      f"SoC: {current_soc:.2f} MWh | "
                      f"Profit: €{cumulative_profit:.2f}")
            
            # Get current order book snapshot
            order_book = order_book_loader.get_order_book_snapshot(current_time)
            
            if not order_book:
                continue
            
            # Convert to time periods (hours from now)
            delivery_times = sorted(order_book.keys())
            
            # Limit horizon
            horizon_limit = current_time + pd.Timedelta(hours=max_horizon_hours)
            delivery_times = [t for t in delivery_times if t <= horizon_limit]
            
            if len(delivery_times) < 2:
                continue
            
            # Create time horizon (use hours as indices)
            time_horizon = list(range(len(delivery_times)))
            
            # Map orders to time periods
            orders = {}
            for i, delivery_time in enumerate(delivery_times):
                orders[i] = order_book[delivery_time]
            
            # Solve optimization
            try:
                if self.use_dp:
                    solution = self.solver.solve(
                        orders, current_soc, time_horizon, 
                        final_soc=None, phi=self.phi
                    )
                else:
                    solution = self.solver.solve(
                        orders, current_soc, time_horizon,
                        committed_power=None, final_soc=None
                    )
                
                if solution['status'] != 'optimal':
                    continue
                
                total_solves += 1
                
                # Extract first-period action
                first_period_soc = solution['soc'].get(0, current_soc)
                first_period_power = solution['power'].get(0, 0.0)
                
                # Update state
                soc_change = first_period_soc - current_soc
                current_soc = first_period_soc
                
                # Track profit (approximate from power and prices)
                period_profit = 0.0
                if 0 in solution.get('trades', {}):
                    for trade in solution['trades'][0]:
                        if trade['is_buy']:
                            period_profit -= trade['price'] * trade['quantity']
                        else:
                            period_profit += trade['price'] * trade['quantity']
                
                # Estimate degradation cost
                if soc_change < 0:
                    deg_cost = self.degradation.compute_degradation_cost(
                        current_soc - soc_change, -soc_change
                    )
                    cumulative_degradation += deg_cost
                
                cumulative_profit += period_profit
                
                # Track metrics
                self.soc_history.append({
                    'time': current_time,
                    'soc': current_soc,
                    'power': first_period_power
                })
                
                self.profit_history.append({
                    'time': current_time,
                    'profit': period_profit,
                    'cumulative': cumulative_profit
                })
                
                self.solve_time_history.append(solution['solve_time'])
                
            except Exception as e:
                print(f"Error at {current_time}: {e}")
                continue
        
        # Compute statistics
        avg_solve_time = np.mean(self.solve_time_history) if self.solve_time_history else 0
        total_solve_time = np.sum(self.solve_time_history)
        
        results = {
            'total_profit': cumulative_profit,
            'total_degradation': cumulative_degradation,
            'net_profit': cumulative_profit - cumulative_degradation,
            'final_soc': current_soc,
            'total_solves': total_solves,
            'avg_solve_time_ms': avg_solve_time * 1000,
            'total_solve_time_s': total_solve_time,
            'soc_history': self.soc_history,
            'profit_history': self.profit_history
        }
        
        return results


# Example usage with CSV data
if __name__ == "__main__":
    import sys
    
    # Check if CSV path provided
    if len(sys.argv) > 1:
        csv_path = sys.argv[1]
    else:
        csv_path = "orderbook_data.csv"  # Default
    
    print("="*70)
    print("Health-Aware BESS Trading with Real Order Book Data")
    print("="*70)
    
    # Initialize parameters
    battery = BatteryParams(
        power_max=10.0,
        power_min=-10.0,
        energy_capacity=10.0,
        eta_charge=0.95,
        eta_discharge=0.95,
        replacement_cost=300000.0,
        num_segments=16
    )
    market = MarketParams(
        trading_fee=0.09,
        min_trading_unit=0.1,
        time_interval=1.0
    )
    degradation = DegradationModel(battery)
    
    # Load order book data
    try:
        loader = OrderBookLoader(csv_path)
    except FileNotFoundError:
        print(f"\nError: Could not find {csv_path}")
        print("Please provide the path to your CSV file as a command line argument:")
        print(f"  python {sys.argv[0]} path/to/your/orderbook.csv")
        sys.exit(1)
    
    # Run comparison: MILP vs DP
    initial_soc = 5.0
    
    print("\n" + "="*70)
    print("Comparison 1: MILP vs DP (on subset of data)")
    print("="*70)
    
    # Test on small subset for MILP (it's slow)
    print("\n[1] Testing MILP (limited to hourly updates)...")
    backtest_milp = RollingIntrinsicBacktest(
        battery, market, degradation, 
        use_dp=False, phi=0.0
    )
    
    # Run on first day only for MILP
    import pandas as pd
    first_day_data = loader.data[
        loader.data['start'] < loader.data['start'].min() + pd.Timedelta(days=1)
    ]
    temp_loader = OrderBookLoader.__new__(OrderBookLoader)
    temp_loader.data = first_day_data
    
    results_milp = backtest_milp.run(
        temp_loader, 
        initial_soc=initial_soc,
        max_horizon_hours=24,
        update_frequency='hourly'
    )
    
    print(f"\nMILP Results:")
    print(f"  Total Profit: €{results_milp['total_profit']:.2f}")
    print(f"  Degradation: €{results_milp['total_degradation']:.2f}")
    print(f"  Net Profit: €{results_milp['net_profit']:.2f}")
    print(f"  Avg Solve Time: {results_milp['avg_solve_time_ms']:.2f} ms")
    print(f"  Total Solves: {results_milp['total_solves']}")
    
    # Test DP on same subset
    print("\n[2] Testing DP (same data, different grid sizes)...")
    for num_states in [11, 51, 101]:
        backtest_dp = RollingIntrinsicBacktest(
            battery, market, degradation,
            use_dp=True, num_states=num_states, phi=0.0
        )
        
        results_dp = backtest_dp.run(
            temp_loader,
            initial_soc=initial_soc,
            max_horizon_hours=24,
            update_frequency='hourly'
        )
        
        print(f"\n  DP-{num_states} Results:")
        print(f"    Total Profit: €{results_dp['total_profit']:.2f}")
        print(f"    Degradation: €{results_dp['total_degradation']:.2f}")
        print(f"    Net Profit: €{results_dp['net_profit']:.2f}")
        print(f"    Avg Solve Time: {results_dp['avg_solve_time_ms']:.2f} ms")
        
        if results_milp['avg_solve_time_ms'] > 0:
            speedup = results_milp['avg_solve_time_ms'] / results_dp['avg_solve_time_ms']
            print(f"    Speedup vs MILP: {speedup:.1f}x")
        
        if results_milp['total_profit'] != 0:
            gap = abs(results_dp['total_profit'] - results_milp['total_profit']) / \
                  abs(results_milp['total_profit']) * 100
            print(f"    Solution Gap: {gap:.2f}%")
    
    # Full backtest with DP
    print("\n" + "="*70)
    print("Full Backtest: DP on Complete Dataset")
    print("="*70)
    
    print("\n[3] Running DP on full data (hourly updates)...")
    backtest_full = RollingIntrinsicBacktest(
        battery, market, degradation,
        use_dp=True, num_states=51, phi=0.0
    )
    
    results_full = backtest_full.run(
        loader,
        initial_soc=initial_soc,
        max_horizon_hours=24,
        update_frequency='hourly'
    )
    
    print(f"\n{'='*70}")
    print("FINAL RESULTS")
    print(f"{'='*70}")
    print(f"Total Profit: €{results_full['total_profit']:,.2f}")
    print(f"Total Degradation: €{results_full['total_degradation']:,.2f}")
    print(f"Net Profit: €{results_full['net_profit']:,.2f}")
    print(f"Final SoC: {results_full['final_soc']:.2f} MWh")
    print(f"Total Optimizations: {results_full['total_solves']}")
    print(f"Avg Solve Time: {results_full['avg_solve_time_ms']:.2f} ms")
    print(f"Total Compute Time: {results_full['total_solve_time_s']:.2f} seconds")
    print(f"{'='*70}\n")
    
    # Export results to CSV
    print("Exporting results...")
    
    # SoC history
    if results_full['soc_history']:
        soc_df = pd.DataFrame(results_full['soc_history'])
        soc_df.to_csv('backtest_soc_history.csv', index=False)
        print(f"  ✓ SoC history saved to: backtest_soc_history.csv")
    
    # Profit history
    if results_full['profit_history']:
        profit_df = pd.DataFrame(results_full['profit_history'])
        profit_df.to_csv('backtest_profit_history.csv', index=False)
        print(f"  ✓ Profit history saved to: backtest_profit_history.csv")
    
    # Summary statistics
    summary = {
        'Metric': [
            'Total Profit (€)',
            'Total Degradation (€)',
            'Net Profit (€)',
            'Initial SoC (MWh)',
            'Final SoC (MWh)',
            'Total Optimizations',
            'Avg Solve Time (ms)',
            'Total Compute Time (s)',
            'Battery Capacity (MWh)',
            'Power Rating (MW)',
            'Round-trip Efficiency',
            'Number of Segments'
        ],
        'Value': [
            f"{results_full['total_profit']:.2f}",
            f"{results_full['total_degradation']:.2f}",
            f"{results_full['net_profit']:.2f}",
            f"{initial_soc:.2f}",
            f"{results_full['final_soc']:.2f}",
            results_full['total_solves'],
            f"{results_full['avg_solve_time_ms']:.2f}",
            f"{results_full['total_solve_time_s']:.2f}",
            f"{battery.energy_capacity:.1f}",
            f"{battery.power_max:.1f}",
            f"{battery.eta_charge * battery.eta_discharge:.3f}",
            battery.num_segments
        ]
    }
    summary_df = pd.DataFrame(summary)
    summary_df.to_csv('backtest_summary.csv', index=False)
    print(f"  ✓ Summary saved to: backtest_summary.csv")
    
    print(f"\n{'='*70}")
    print("Backtest Complete!")
    print(f"{'='*70}\n")